In [2]:
%cd ~/code/entityrepresentations/

/home/morand/code/entityrepresentations


/home/morand/reimplems/taskVectorsHendel/env/lib/python3.10/site-packages/IPython/core/magics/osm.py:393: UserWarning: This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})
/home/morand/reimplems/taskVectorsHendel/env/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
import transformers
import transformer_lens as tl 
from transformer_lens import HookedTransformer, patching
from importlib import reload
import torch, os, gc, sys, pathlib
import torch.nn as nn
from tqdm import tqdm
#our own code
import utils 

reload(utils)
from LabelExtractor import eval_model, infer_entities
reload(sys.modules['LabelExtractor'])
from processResults import *
reload(sys.modules['processResults'])



import circuitsvis as cv
import plotly.io as pio
# Plotly needs a different renderer for VSCode/Notebooks vs Colab argh
pio.renderers.default = "notebook_connected"
print(f"Using renderer: {pio.renderers.default}")
# Testing that the library works
cv.examples.hello("Victor")


Using renderer: notebook_connected


## Params 

In [4]:
# Load a model (eg GPT-2 Small)
model_name = "bloom-3b" #crashed
model_name = "meta-llama/Meta-Llama-3-8B" # ?
model_name = "gpt2-small" # 117M ok
model_name = "pythia-2.8b"#ok !
model_name = "gpt2-xl" # 1.5B ok
model_name = "mistralai/Mistral-7B-v0.1" # ok JZ a100 8cpus | 
model_name = "gpt2-large" # 774M ok
model_name = "gpt2-medium" # 302M ok
model_name = "phi-2" # 2,5B ok 12cpus nope, gpu 24cpus ok
model_name = "phi-1_5"  # 1.5B ok

dataset_name= "WebNLG"
dataset_name= "TACRED"

with_context = True

xp_path = pathlib.Path.home() / "experiments_JeanZay"

# load results
xp_path = pathlib.Path.home() / "experiments_JeanZay"
#get all experiments directories
xp_paths= os.listdir(xp_path / "jobs")
jobs_path = xp_path / "jobs" / xp_paths[0]
#get jobs
results = loadResults(jobs_path)

100%|████████████████████████████████████████████████████████████████████████████| 550/550 [00:02<00:00, 184.94it/s]


## Load Model

In [5]:

#check if model variable exists
if not 'model' in locals():
    model = HookedTransformer.from_pretrained(
                                    model_name, 
                                    trust_remote_code = True, 
                                    low_cpu_mem_usage = True, 
                                    device_map='auto',
                                    move_to_device=False,
                                    fold_ln=False,
                                    fold_value_biases=False,
                                    center_writing_weights=False,
                                    center_unembed=False,
                                    )
    dim = model.QK.shape[-1]
print(model)
model.eval()
model = model.cuda()

/home/morand/reimplems/taskVectorsHendel/env/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning:

`resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.



Loaded pretrained model phi-1_5 into HookedTransformer
HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (blocks): ModuleList(
    (0-23): 24 x TransformerBlock(
      (ln1): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
        (hook_rot_k): HookPoint()
        (hook_rot_q): HookPoint()
      )
      (mlp): MLP(
        (hook_pre): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_attn_o

In [5]:
models = ['gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl', 'distilgpt2', 'facebook/opt-125m', 'facebook/opt-1.3b', 'facebook/opt-2.7b', 'facebook/opt-6.7b', 'facebook/opt-13b', 'facebook/opt-30b', 'facebook/opt-66b', 'EleutherAI/gpt-neo-125M', 'EleutherAI/gpt-neo-1.3B', 'EleutherAI/gpt-neo-2.7B', 'EleutherAI/gpt-j-6B', 'EleutherAI/gpt-neox-20b', 'stanford-crfm/alias-gpt2-small-x21', 'stanford-crfm/battlestar-gpt2-small-x49', 'stanford-crfm/caprica-gpt2-small-x81', 'stanford-crfm/darkmatter-gpt2-small-x343', 'stanford-crfm/expanse-gpt2-small-x777', 'stanford-crfm/arwen-gpt2-medium-x21', 'stanford-crfm/beren-gpt2-medium-x49', 'stanford-crfm/celebrimbor-gpt2-medium-x81', 'stanford-crfm/durin-gpt2-medium-x343', 'stanford-crfm/eowyn-gpt2-medium-x777', 'EleutherAI/pythia-14m', 'EleutherAI/pythia-31m', 'EleutherAI/pythia-70m', 'EleutherAI/pythia-160m', 'EleutherAI/pythia-410m', 'EleutherAI/pythia-1b', 'EleutherAI/pythia-1.4b', 'EleutherAI/pythia-2.8b', 'EleutherAI/pythia-6.9b', 'EleutherAI/pythia-12b', 'EleutherAI/pythia-70m-deduped', 'EleutherAI/pythia-160m-deduped', 'EleutherAI/pythia-410m-deduped', 'EleutherAI/pythia-1b-deduped', 'EleutherAI/pythia-1.4b-deduped', 'EleutherAI/pythia-2.8b-deduped', 'EleutherAI/pythia-6.9b-deduped', 'EleutherAI/pythia-12b-deduped', 'EleutherAI/pythia-70m-v0', 'EleutherAI/pythia-160m-v0', 'EleutherAI/pythia-410m-v0', 'EleutherAI/pythia-1b-v0', 'EleutherAI/pythia-1.4b-v0', 'EleutherAI/pythia-2.8b-v0', 'EleutherAI/pythia-6.9b-v0', 'EleutherAI/pythia-12b-v0', 'EleutherAI/pythia-70m-deduped-v0', 'EleutherAI/pythia-160m-deduped-v0', 'EleutherAI/pythia-410m-deduped-v0', 'EleutherAI/pythia-1b-deduped-v0', 'EleutherAI/pythia-1.4b-deduped-v0', 'EleutherAI/pythia-2.8b-deduped-v0', 'EleutherAI/pythia-6.9b-deduped-v0', 'EleutherAI/pythia-12b-deduped-v0', 'EleutherAI/pythia-160m-seed1', 'EleutherAI/pythia-160m-seed2', 'EleutherAI/pythia-160m-seed3', 'NeelNanda/SoLU_1L_v9_old', 'NeelNanda/SoLU_2L_v10_old', 'NeelNanda/SoLU_4L_v11_old', 'NeelNanda/SoLU_6L_v13_old', 'NeelNanda/SoLU_8L_v21_old', 'NeelNanda/SoLU_10L_v22_old', 'NeelNanda/SoLU_12L_v23_old', 'NeelNanda/SoLU_1L512W_C4_Code', 'NeelNanda/SoLU_2L512W_C4_Code', 'NeelNanda/SoLU_3L512W_C4_Code', 'NeelNanda/SoLU_4L512W_C4_Code', 'NeelNanda/SoLU_6L768W_C4_Code', 'NeelNanda/SoLU_8L1024W_C4_Code', 'NeelNanda/SoLU_10L1280W_C4_Code', 'NeelNanda/SoLU_12L1536W_C4_Code', 'NeelNanda/GELU_1L512W_C4_Code', 'NeelNanda/GELU_2L512W_C4_Code', 'NeelNanda/GELU_3L512W_C4_Code', 'NeelNanda/GELU_4L512W_C4_Code', 'NeelNanda/Attn_Only_1L512W_C4_Code', 'NeelNanda/Attn_Only_2L512W_C4_Code', 'NeelNanda/Attn_Only_3L512W_C4_Code', 'NeelNanda/Attn_Only_4L512W_C4_Code', 'NeelNanda/Attn-Only-2L512W-Shortformer-6B-big-lr', 'NeelNanda/SoLU_1L512W_Wiki_Finetune', 'NeelNanda/SoLU_4L512W_Wiki_Finetune', 'ArthurConmy/redwood_attn_2l', 'llama-7b-hf', 'llama-13b-hf', 'llama-30b-hf', 'llama-65b-hf', 'meta-llama/Llama-2-7b-hf', 'meta-llama/Llama-2-7b-chat-hf', 'meta-llama/Llama-2-13b-hf', 'meta-llama/Llama-2-13b-chat-hf', 'meta-llama/Llama-2-70b-chat-hf', 'CodeLlama-7b-hf', 'CodeLlama-7b-Python-hf', 'CodeLlama-7b-Instruct-hf', 'meta-llama/Meta-Llama-3-8B', 'meta-llama/Meta-Llama-3-8B-Instruct', 'meta-llama/Meta-Llama-3-70B', 'meta-llama/Meta-Llama-3-70B-Instruct', 'Baidicoot/Othello-GPT-Transformer-Lens', 'bert-base-cased', 'roneneldan/TinyStories-1M', 'roneneldan/TinyStories-3M', 'roneneldan/TinyStories-8M', 'roneneldan/TinyStories-28M', 'roneneldan/TinyStories-33M', 'roneneldan/TinyStories-Instruct-1M', 'roneneldan/TinyStories-Instruct-3M', 'roneneldan/TinyStories-Instruct-8M', 'roneneldan/TinyStories-Instruct-28M', 'roneneldan/TinyStories-Instruct-33M', 'roneneldan/TinyStories-1Layer-21M', 'roneneldan/TinyStories-2Layers-33M', 'roneneldan/TinyStories-Instuct-1Layer-21M', 'roneneldan/TinyStories-Instruct-2Layers-33M', 'stabilityai/stablelm-base-alpha-3b', 'stabilityai/stablelm-base-alpha-7b', 'stabilityai/stablelm-tuned-alpha-3b', 'stabilityai/stablelm-tuned-alpha-7b', 'mistralai/Mistral-7B-v0.1', 'mistralai/Mistral-7B-Instruct-v0.1', 'mistralai/Mixtral-8x7B-v0.1', 'mistralai/Mixtral-8x7B-Instruct-v0.1', 'bigscience/bloom-560m', 'bigscience/bloom-1b1', 'bigscience/bloom-1b7', 'bigscience/bloom-3b', 'bigscience/bloom-7b1', 'bigcode/santacoder', 'Qwen/Qwen-1_8B', 'Qwen/Qwen-7B', 'Qwen/Qwen-14B', 'Qwen/Qwen-1_8B-Chat', 'Qwen/Qwen-7B-Chat', 'Qwen/Qwen-14B-Chat', 'Qwen/Qwen1.5-0.5B', 'Qwen/Qwen1.5-0.5B-Chat', 'Qwen/Qwen1.5-1.8B', 'Qwen/Qwen1.5-1.8B-Chat', 'Qwen/Qwen1.5-4B', 'Qwen/Qwen1.5-4B-Chat', 'Qwen/Qwen1.5-7B', 'Qwen/Qwen1.5-7B-Chat', 'Qwen/Qwen1.5-14B', 'Qwen/Qwen1.5-14B-Chat', 'microsoft/phi-1', 'microsoft/phi-1_5', 'microsoft/phi-2', 'google/gemma-2b', 'google/gemma-7b', 'google/gemma-2b-it', 'google/gemma-7b-it', '01-ai/Yi-6B', '01-ai/Yi-34B', '01-ai/Yi-6B-Chat', '01-ai/Yi-34B-Chat']

#find all phi models
[m for m in models if 'pythia' in m]


['EleutherAI/pythia-14m',
 'EleutherAI/pythia-31m',
 'EleutherAI/pythia-70m',
 'EleutherAI/pythia-160m',
 'EleutherAI/pythia-410m',
 'EleutherAI/pythia-1b',
 'EleutherAI/pythia-1.4b',
 'EleutherAI/pythia-2.8b',
 'EleutherAI/pythia-6.9b',
 'EleutherAI/pythia-12b',
 'EleutherAI/pythia-70m-deduped',
 'EleutherAI/pythia-160m-deduped',
 'EleutherAI/pythia-410m-deduped',
 'EleutherAI/pythia-1b-deduped',
 'EleutherAI/pythia-1.4b-deduped',
 'EleutherAI/pythia-2.8b-deduped',
 'EleutherAI/pythia-6.9b-deduped',
 'EleutherAI/pythia-12b-deduped',
 'EleutherAI/pythia-70m-v0',
 'EleutherAI/pythia-160m-v0',
 'EleutherAI/pythia-410m-v0',
 'EleutherAI/pythia-1b-v0',
 'EleutherAI/pythia-1.4b-v0',
 'EleutherAI/pythia-2.8b-v0',
 'EleutherAI/pythia-6.9b-v0',
 'EleutherAI/pythia-12b-v0',
 'EleutherAI/pythia-70m-deduped-v0',
 'EleutherAI/pythia-160m-deduped-v0',
 'EleutherAI/pythia-410m-deduped-v0',
 'EleutherAI/pythia-1b-deduped-v0',
 'EleutherAI/pythia-1.4b-deduped-v0',
 'EleutherAI/pythia-2.8b-deduped-v0',

## Load the data

In [6]:
import torch
import numpy as np
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset

max_ent_length = 20
max_dev_length = 1000

print(f"loading data from {dataset_name} ...")
if dataset_name.lower() == "webnlg":
    dataset = load_dataset("web_nlg", "release_v3.0_en", trust_remote_code=True)

    #optionnal, filter categories from datset
    cat = ['Food'] #WebNLG Categories to remove
    # cat = None
    if cat :
        dataset["train"] = [item for item in dataset["train"] if item["category"] not in cat]
        dataset["dev"] = [item for item in dataset["dev"] if item["category"] not in cat]
        dataset["test"] = [item for item in dataset["test"] if item["category"] not in cat]

    # Create dataset instances
    train_dataset = utils.WebNLGDataset(dataset['train'], max_ent_length=max_ent_length)
    dev_dataset = utils.WebNLGDataset(dataset['dev'], max_ent_length=max_ent_length)
    test_dataset = utils.WebNLGDataset(dataset['test'], max_ent_length=max_ent_length)

elif dataset_name.lower() == "tacred":
    dataset = load_dataset("AmirLayegh/tacred_text_label")
    train_dataset = utils.TacredDataset(dataset["train"], max_ent_length=max_ent_length, max_length=200)
    dev_dataset = utils.TacredDataset(dataset["test"], max_ent_length=max_ent_length, max_length=200)
    test_dataset = utils.TacredDataset(dataset["validation"], max_ent_length=max_ent_length, max_length=200)

else: 
    # unknown Dataset 
    raise NotImplementedError("dataset Name must be either 'webnlg' or 'tacred'")


#limit the number of samples for testing
print(f"initial train dataset size: {len(dev_dataset.data)}, truncating to {max_dev_length}")
dev_dataset.data = list(np.random.choice(dev_dataset.data, max_dev_length, replace=False))

loading data from TACRED ...
initial train dataset size: 11340, truncating to 1000


In [7]:
dataset.keys()

dict_keys(['train', 'validation', 'test'])

In [8]:
print("train length:", len(train_dataset))
print("dev length:", len(dev_dataset))
print("test length:", len(test_dataset))
print("ex sample:", dev_dataset[np.random.randint(len(dev_dataset))])

train length: 51506
dev length: 1000
test length: 17056
ex sample: {'text': 'Jamie Leigh Jones, of Texas, sued the companies after she says she was raped while working for KBR in Baghdad in 2005.', 'entity': 'Jamie Leigh Jones', 'id': 1151}


## Test TaskVecs

In [46]:
fileName = "./checkpoints/TaskVec_gpt2-small_l8_e50.pth"
fileName = "./checkpoints/TaskVec_mistral-7B_l17_e20.pth"
fileName = "./LabelExtractor_gpt2-medium_l12_e9.pth"
fileName = "./checkpoints/contextLabelExtractor_phi-2_l15_e6.pth"
fileName = "./checkpoints/LabelExtractor_phi-2_l15_e20.pth"
fileName = "./checkpoints/contextTaskVec_gpt2-medium_l13_e20.pth"
layer = 10
fileName = get_taskVec(results, model_name, layer=layer, dataset_name=dataset_name, with_context = with_context)

TaskVec = torch.load(fileName)
print("TaskVec loaded from ", fileName)


found 1 results for layer 10 of phi-1_5 with context on TACRED
found ['TaskVec_phi-1_5_l10_e20.pth'] 
TaskVec loaded from  /home/morand/experiments_JeanZay/jobs/labelextractor.learnlabelextractor/a0bb73f972aa692249b7cb4a41704b416da55d29aebe44fd2e8b610bbdadb10b/TaskVec_phi-1_5_l10_e20.pth


## Retrieve Entity Labels 

In [47]:
#funny widget to select a word 
# Split the text into words
ind = np.random.randint(len(test_dataset))
# ind = 7616
context = test_dataset[ind]["text"]
# context = " _ > Agnes Kant"
# context = "Alan Bean was an American astronaut, born on March 15, 1932 in Wheeler, Texas. He received a Bachelor of Science degree at the University of Texas at Austin in 1955 and was chosen by NASA in 1963. _ > Alan Bean"
words = model.to_str_tokens(context)
print(len(words))
cv.tokens.colored_tokens(words, words)


23


In [48]:
#compute whole cache
print("computing cache ...")
#get whole hidden states
_ , cache = model.run_with_cache(context)
repr = cache[tl.utils.get_act_name("resid_post", layer, "")][0,:,:] # 1 x n_tokens x dim
repr = repr.detach().cuda()
print(repr.shape)
data = [
    {   
        "id": i,
        "text": context,
        "representation": repr[i],
        "tok": words[i],
    }
    for i in range(len(words))
]
infer_entities(model, TaskVec, data, with_context=with_context)

computing cache ...
torch.Size([23, 2048])


### Raw display

In [30]:
print(data[0]["text"])
for it in data:
    print(f"Token: {it['tok'].center(10)} | Gen:  {it['inferred']}" )

I was called to come up with -LRB- Spector -RRB- and -LRB- songwriters -RRB- Jeff Barry and Ellie Greenwich.
Token: <|endoftext|> | Gen:  Ellie Greenwich
Token:     I      | Gen:  I
Token:     was    | Gen:  
Token:   called   | Gen:  called
Token:     to     | Gen:  Spector
Token:    come    | Gen:  Ellie Greenwich
Token:     up     | Gen:  Spector -RRB-
Token:    with    | Gen:  Spector -RRB-
Token:      -     | Gen:  
Token:     L      | Gen:  Ellie Greenwich
Token:     RB     | Gen:  LRB-
Token:     -      | Gen:  Ellie Greenwich
Token:    Spect   | Gen:  Spector
Token:     or     | Gen:  Ellie Greenwichor
Token:      -     | Gen:  Ellie Greenwich
Token:     RR     | Gen:  Ellie Greenwich
Token:     B      | Gen:  Ellie Greenwich
Token:     -      | Gen:  Ellie Greenwich
Token:     and    | Gen:  Ellie Greenwich
Token:      -     | Gen:  
Token:     L      | Gen:  L
Token:     RB     | Gen:  LRB
Token:     -      | Gen:  Ellie Greenwich
Token:    song    | Gen:  song
Token:  writer

### Circuitvis
smoother visualization

In [49]:
html = cv.tokens.colored_tokens(words, [it['inferred'] for it in data])
html.cdn_src = html.cdn_src.replace("margin: 15px", "margin: 50px")
html

### Old IPywidget code

In [74]:
from IPython.display import display
import ipywidgets as widgets
reload(widgets)

prompt = train_dataset[np.random.randint(len(train_dataset))]["text"]
# prompt = " _ > Agnes Kant"
# prompt = "Alan Bean was an American astronaut, born on March 15, 1932 in Wheeler, Texas. He received a Bachelor of Science degree at the University of Texas at Austin in 1955 and was chosen by NASA in 1963. _ > Alan Bean"
words = model.to_str_tokens(prompt)
print(words)
print(len(words))

# Callback function to update selected word
def on_button_click(b):
    selected_word_label.value = f"Selected Word: {b.description}"
    layer = dropdown.value
    repr = utils.get_representation(model,
                                layer=layer,
                                tokens=model.to_tokens(prompt),
                                token_inds=torch.tensor([b.id]),
                                verbose=True)
    #change button color
    b.style.button_color = 'lightgreen'
    #change all others to default
    for button in buttons:
        if button.id != b.id:
            button.style.button_color = 'white'
    data = [{
        "id":0,
        "representation": repr,
        "text": prompt
        }]
    infer_entities(model, TaskVec, data, with_context=with_context)
    print("generation:", data[0]["inferred"])

# Buttons for each word
buttons = []
class clickableToken(widgets.Button):
    def __init__(self, id:int, description:str, **kwargs):
        super().__init__(description=description, **kwargs)
        self.description = description
        self.id = id
        self.on_click(on_button_click)

for ind, token in enumerate(words):
    buttons.append(
        clickableToken(
            id=ind, 
            description=token,
            layout=widgets.Layout(width='auto', margin='2px', padding='0 5px')
        ))
    

# Box widget with flexible wrapping
button_box = widgets.Box(
    children=buttons,
    layout=widgets.Layout(display='flex', flex_flow='row wrap', align_items='center')
)
# Dropdown widget for selecting an integer
dropdown = widgets.Dropdown(
    options=[(f"layer {i}", i) for i in range(len(model.blocks)-1, -1, -1)],
    description='Select a layer:',
    disabled=False,    
)
dropdown.value = layer
def on_checkbox_change(change):
    global with_context
    with_context = change['new']

# Create the checkbox widget
with_context_checkbox = widgets.Checkbox(
    value=with_context,
    description='Generate with context',
    disabled=False
)
# Link the function to the checkbox change event
with_context_checkbox.observe(on_checkbox_change, names='value')

# Label to display the selected word
selected_word_label = widgets.Label()

# # clean previous display
# display.clear_output() #works ?? 

# Create a title using HTML widget
title = widgets.HTML(value="<h3>Select a Token to generate from:</h3>")

# Display widgets
display(title)
display(button_box)
#display checkbox  and dropdown side by side
display(widgets.HBox([with_context_checkbox, dropdown]))
display(selected_word_label)

['<|endoftext|>', 'Ab', 'rams', ',', ' who', ' has', ' been', ' filling', '-', 'in', ' as', ' host', ' of', ' the', ' 9', ' p', '.', 'm', '.', ' hour', ' for', ' the', ' past', ' three', ' months', ',', ' will', ' continue', ' to', ' anchor', ' that', ' hour', ' as', ' host', ' of', ' ``', ' Live', ' with', ' Dan', ' Abrams', ',', " ''", ' Monday', '-', 'Thursday', ' at', ' 9', ' p', '.', 'm', '.', ' ET', '.']
53


HTML(value='<h3>Select a Token to generate from:</h3>')

Box(children=(clickableToken(description='<|endoftext|>', layout=Layout(margin='2px', padding='0 5px', width='…

Label(value='')